## 6. Filter Bank Common Spatial Patterns (FBCSP)

The baseline CSP+LDA pipeline uses a single 8–30 Hz band. But motor-imagery
signal sits in different frequencies for different people — some in the mu band
(~8–13 Hz), some in beta (~13–30 Hz). A single wide band mixes the informative
sub-band with uninformative ones, diluting the signal.

**FBCSP** (Ang et al., BCI Competition IV winner) addresses this by analyzing
multiple narrow frequency bands and letting the data decide which matter:

1. **Filter bank** — split the signal into narrow bands (here, 7 bands of 4 Hz
   width from 4–32 Hz), covering theta, mu, and beta.
2. **CSP per band** — run CSP *separately* on each band, so each frequency range
   gets its own spatial filters optimized for that band.
3. **Feature selection** — pool the features from all bands, then select the most
   discriminative ones (via mutual information). This automatically identifies
   *which bands carry the signal for this specific subject* — the key advantage
   over single-band CSP.
4. **Classification** — LDA on the selected features.

The core idea: instead of guessing one frequency band, FBCSP tries many and lets
the data pick. This directly targets the subject-to-subject variability found
earlier, where different subjects' signals lived in different frequency ranges.

**Future directions (deep learning):** FBCSP's multi-band spatial-filtering idea
has neural-network descendants worth benchmarking later — **FBCNet** (a deep
learning version of FBCSP's multi-band structure) and **EEGNet** (a compact CNN
noted in reviews for its generalizability). These need more data than the
classical methods, so they are planned as later comparisons once the classical
pipeline and hardware are working.

In [7]:
# Imports for FBCSP work
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci
from mne.decoding import CSP
from pathlib import Path

# scikit-learn: classifier, pipeline, cross-validation, feature selection
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif

print("mne:", mne.__version__)
print("Imports OK")

mne: 1.12.1
Imports OK


In [8]:
# Shared configuration
data_dir = Path.home() / "projects" / "bci" / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

# Standard electrode-position montage, reused throughout
montage = mne.channels.make_standard_montage("standard_1005")

# The filter bank: 7 narrow bands (4 Hz wide) covering theta/mu/beta
filter_bank = [
    (4, 8), (8, 12), (12, 16), (16, 20),
    (20, 24), (24, 28), (28, 32),
]

print("Data dir:", data_dir)
print(f"Filter bank: {len(filter_bank)} bands ->", filter_bank)

Data dir: /home/beecki303/projects/bci/data/raw
Filter bank: 7 bands -> [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32)]


In [9]:
def load_subject_raw(subject, runs=(4, 8, 12)):
    """Download + load + standardize a subject's motor imagery runs."""
    fnames = eegbci.load_data(subjects=[subject], runs=list(runs),
                              path=str(data_dir), verbose=False)
    raws = [mne.io.read_raw_edf(f, preload=True, verbose=False) for f in fnames]
    raw = mne.concatenate_raws(raws, verbose=False)
    eegbci.standardize(raw)
    raw.set_montage(montage, verbose=False)
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    return raw, events, event_id

In [10]:
def make_band_epochs(raw_source, band, events, event_id):
    """Bandpass to one filter-bank band, then epoch the imagery window."""
    lo, hi = band
    r = raw_source.copy()
    r.notch_filter(60.0, picks='eeg', verbose=False)
    r.filter(lo, hi, picks='eeg', verbose=False)
    ep = mne.Epochs(r, events,
                    event_id={'T1': event_id['T1'], 'T2': event_id['T2']},
                    tmin=0.5, tmax=2.5, baseline=None,
                    picks='eeg', preload=True, verbose=False)
    return ep

def fbcsp_features(raw_source, events, event_id, n_csp=4):
    """Run CSP on each band, stack log-variance features across all bands."""
    band_features = []
    for band in filter_bank:
        ep = make_band_epochs(raw_source, band, events, event_id)
        Xb = ep.get_data(copy=False)
        yb = ep.events[:, -1]
        csp = CSP(n_components=n_csp, reg=None, log=True, norm_trace=False)
        feats = csp.fit_transform(Xb, yb)
        band_features.append(feats)
    X_fb = np.hstack(band_features)
    return X_fb, yb

In [11]:
# Verify setup: load subject 2 and build the FBCSP feature matrix
raw_s, evs, ev_id = load_subject_raw(2)
X_fb, y_fb = fbcsp_features(raw_s, evs, ev_id, n_csp=4)

print("FBCSP feature matrix shape:", X_fb.shape)
print(f"  = {len(filter_bank)} bands x 4 CSP components = {X_fb.shape[1]} features")
print("Labels shape:", y_fb.shape)

Computing rank from data with rank=None
    Using tolerance 7.9e-05 (2.2e-16 eps * 64 dim * 5.5e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 8.5e-05 (2.2e-16 eps * 64 dim * 6e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 7.5e-05 (2.2e-16 eps * 64 dim * 5.3e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIR